In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import shapiq
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from tqdm.asyncio import tqdm

In [ ]:
methyl_data = pd.read_csv('../data/TCGA/omics_data/preprocessed/all_patients/TCGA_PDAC_Methylation.csv.csv', index_col=0)
final_features = methyl_data[['cg00839579', 'cg06785999', 'cg07095230']]
final_clusters = pd.read_csv('../data/patient_clusters.csv', index_col=0)
final_clusters = {k: [x for x in v if pd.notna(x)] for k, v in final_clusters.items()}
methyl_data['Cluster'] = None
for cluster_num, patients in final_clusters.items():
    cluster_label = int(cluster_num.split('_')[-1])+1
    methyl_data.loc[methyl_data.index.isin(patients), 'Cluster'] = cluster_label
shap_df = methyl_data.dropna(subset=['Cluster'])
x_data = shap_df.iloc[:, :-1]
y_data = shap_df.iloc[:, -1]
feature_names = list(x_data.columns)
n_features = len(feature_names)
x_data, y_data = x_data.values, y_data.values
print(f"{n_features} features in the dataset:", feature_names)

In [ ]:
from sklearn.metrics import matthews_corrcoef

x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.2, random_state=42)
model = RandomForestClassifier(random_state=42)
model.fit(x_train, y_train)

# evaluate the model using Matthews Correlation Coefficient
y_pred = model.predict(x_test)
mcc = matthews_corrcoef(y_test, y_pred)
print(f"Matthews Correlation Coefficient: {mcc}")

In [ ]:
instance_id = 7
x_explain = x_test[instance_id]
y_true = y_test[instance_id]
y_pred = model.predict(x_explain.reshape(1, -1))[0]
print(f"Instance {instance_id}, True Value: {y_true}, Predicted Value: {y_pred}")
for i, feature in enumerate(feature_names):
    print(f"{feature}: {x_explain[i]}")

In [ ]:
explanations = []
explainer = shapiq.TreeExplainer(model=model, max_order=2, index="k-SII")
for instance_id in tqdm(range(20)):
    x_explain = x_test[instance_id]
    si = explainer.explain(x=x_explain)
    explanations.append(si)
ax = shapiq.plot.bar_plot(explanations, feature_names=feature_names, show=False, abbreviate=False)
fig = ax.get_figure()
fig.savefig('FIGURES/shapiq_barplot.svg', bbox_inches="tight")
plt.show()